<a href="https://colab.research.google.com/github/Azidian/Inteligencia_Artificial/blob/main/Componente_practico/Lecture_06/Challenge/02_training_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [ ]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [ ]:
feature_columns = contract["features"]

# Preprocesador para árboles: imputación por mediana (no necesitan escalado)
preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns)
    ]
)

# Preprocesador para modelos sensibles a la escala: imputación + StandardScaler
preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_columns)
    ]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si se ajusta el scaler con todo el conjunto de entrenamiento antes de la validación cruzada la media y la desviación estándar ya incluyen información de los datos que luego se son usados como validación en cada fold. Eso es una fuga de información, el modelo "ve" indirectamente datos que debería desconocer y el resultado de la CV sale demasiado optimista. Al dejar el scaler dentro del `Pipeline`, en cada fold se ajusta solo con los datos de entrenamiento de ese fold y luego se aplica al fold de validación, que es lo que pasaría en producción con datos nuevos.

## 3. Modelos candidatos

In [ ]:
models = {
    "DecisionTree": Pipeline([
        ("prep", clone(preprocess_tree)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("prep", clone(preprocess_scaled)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("prep", clone(preprocess_scaled)),
        ("model", KNeighborsClassifier()),
    ]),
}
list(models)

['DecisionTree', 'LogisticRegression', 'KNN']

## 4. Comparación inicial con validación cruzada

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring="accuracy",
        return_train_score=True
    )
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
initial_comparison.round(4)

,model,train_accuracy,val_accuracy,val_std
0,LogisticRegression,1.0000,0.9791,0.0277
1,KNN,0.9771,0.9507,0.0353
2,DecisionTree,1.0000,0.9022,0.0586


## 5. Optimización de hiperparámetros

In [ ]:
param_grids = {
    "DecisionTree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 3, 5],
    },
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipe in models.items():
    gs = GridSearchCV(
        pipe, param_grids[name],
        cv=cv, scoring="accuracy",
        return_train_score=True
    )
    gs.fit(X_train, y_train)
    searches[name] = gs

    best = gs.best_index_
    rows.append({
        "model": name,
        "best_cv_accuracy": gs.best_score_,
        "cv_std": gs.cv_results_["std_test_score"][best],
        "train_accuracy": gs.cv_results_["mean_train_score"][best],
        "best_params": gs.best_params_,
    })

tuned_comparison = pd.DataFrame(rows).sort_values("best_cv_accuracy", ascending=False).reset_index(drop=True)
tuned_comparison

,model,best_cv_accuracy,cv_std,train_accuracy,best_params
0,LogisticRegression,0.979064,0.027713,1.000000,{'model__C': 1}
1,KNN,0.957882,0.040520,0.964773,"{'model__n_neighbors': 7, 'model__weights': 'u..."
2,DecisionTree,0.902217,0.058617,0.998246,"{'model__max_depth': 4, 'model__min_samples_le..."


## 6. Selección y persistencia del campeón

In [ ]:
# Selección: mayor best_score_ de CV (desempate: menor desviación estándar)
champion_name = max(
    searches,
    key=lambda n: (round(searches[n].best_score_, 6),
                   -searches[n].cv_results_["std_test_score"][searches[n].best_index_])
)
champion = searches[champion_name].best_estimator_

print("Campeón:", champion_name)
print("CV accuracy:", round(searches[champion_name].best_score_, 4))
print("Parámetros:", searches[champion_name].best_params_)

# Guardar pipeline completo
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# Guardar metadatos
training_metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv_std": float(searches[champion_name].cv_results_["std_test_score"][searches[champion_name].best_index_]),
    "cv_folds": 5,
    "random_state": RANDOM_STATE,
    "n_train": int(len(X_train)),
    "features": feature_columns,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2), encoding="utf-8"
)

# Guardar resultados de CV
cv_results = tuned_comparison.copy()
cv_results["best_params"] = cv_results["best_params"].astype(str)
cv_results.merge(
    initial_comparison[["model", "val_accuracy"]].rename(columns={"val_accuracy": "initial_cv_accuracy"}),
    on="model"
).to_csv(REPORTS_DIR / "cv_results.csv", index=False)

Campeón: LogisticRegression
CV accuracy: 0.9791
Parámetros: {'model__C': 1}


In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

El modelo definitivo es la Regresión Logística (con C = 1), ya que logró el mejor promedio de aciertos (0.979) durante las pruebas. Las razones principales son:

Rinde mejor: Fue el modelo con mayor precisión, superando claramente a KNN (0.958) y al Árbol de Decisión (0.902).

Es más estable: Sus resultados variaron muy poco entre las distintas rondas de prueba (tuvo la menor desviación estándar, 0.028). Esto significa que es un modelo consistente en el que se puede confiar.

No memoriza, aprende: El Árbol de Decisión casi sacó nota perfecta en entrenamiento (0.998) pero le fue mal en validación (0.902), es decir, se sobreajustó y memorizó los datos. La Regresión Logística sacó 1.0 en entrenamiento y se mantuvo súper alta en validación (0.979), demostrando que realmente generaliza bien.

Es sencillo y rápido: Solo necesita ajustar un valor clave (C), por lo que se entrena y funciona rapidísimo. KNN es más pesado porque tiene que guardar todos los datos y calcular distancias cada vez que predice algo nuevo.

Es fácil de entender: Nos permite ver fácilmente qué características del vino son las más importantes para clasificarlo, algo mucho más transparente que intentar seguir las reglas de un árbol enorme o los vecinos de KNN.
> En resumen, la Regresión Logística nos da el mejor equilibrio: es precisa, no falla de forma brusca y es fácil de explicar. Además, esta decisión se tomó de forma limpia, usando solo las pruebas de validación cruzada y sin tocar los datos de test.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.